In [ ]:
import os
import pandas as pd
import geopandas as gpd
import rasterio
import re
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# user-defined paths
base_dir = '../../../../datos-notebooks/3.global-data-cba/'
ground_data_name = 'ground_data_160125.csv'
ground_data_filtered_name = 'ground_data_filtered.gpkg'
ground_data_filtered_fixed_name = 'ground_data_filtered_fixed.gpkg' # ubication fixed manually in qgis

In [ ]:
# derived paths
ground_data_path = os.path.join(base_dir, 'MRT-prediction/ground_data', ground_data_name)
ground_data_filtered_path = os.path.join(base_dir, 'MRT-prediction/', ground_data_filtered_name)
ground_data_filtered_fixed_path = os.path.join(base_dir, 'MRT-prediction/', ground_data_filtered_fixed_name)
mrt_path = os.path.join(base_dir, 'MRT-prediction/MRT_1m.tif')

#### fixing csv and georreferencing data

In [ ]:
df = pd.read_csv(ground_data_path, sep=";")
df = df[['Fecha', 'Hora', 'Ubicación', 'LAT', 'LON', 'MRT °C', 'SIN SOL / PLENO SOL', 'Descripción']]
parts = df['LAT'].str.extract(r"(\d{1,2})°(\d{1,2})'(\d{1,2})\"?S")
df['lat_dd'] = -1 * (parts[0].astype(float) + parts[1].astype(float)/60 + parts[2].astype(float)/3600)
parts = df['LON'].str.extract(r"(\d{1,3})°(\d{1,2})'(\d{1,2})\"?W")
df['lon_dd'] = -1 * (parts[0].astype(float) + parts[1].astype(float)/60 + parts[2].astype(float)/3600)



In [ ]:
# create gdf from decimal
gdf = gpd.GeoDataFrame(
    df,
    geometry = gpd.points_from_xy(df['lon_dd'], df['lat_dd']),
    crs = "EPSG:4326"
)

# read raster crs and project to same crs
with rasterio.open(mrt_path) as src:
    mrt_crs = src.crs
    gdf_proj = gdf.to_crs(mrt_crs)
    

gdf_proj['Hora'] = pd.to_datetime(gdf_proj['Hora'], format="%H:%M", errors="coerce") # convert Hora column to time

gdf_f = gdf_proj[ # filtering by hour
    (gdf_proj['Hora'].dt.time >= pd.to_datetime("10:30").time()) &
    (gdf_proj['Hora'].dt.time <= pd.to_datetime("11:30").time())
]

# export gpkg

gdf_f.to_file(ground_data_filtered_path, driver="GPKG")


### validation


In [ ]:
points = gpd.read_file(ground_data_filtered_fixed_path)
len(points)

In [ ]:
# read raster crs and sample values
with rasterio.open(mrt_path) as src:
    mrt_crs = src.crs
    points_proj = points.to_crs(mrt_crs)
    samples = list(src.sample(zip(points_proj.geometry.x, points_proj.geometry.y)))
    points_proj['MRT_predicted'] = [s[0] for s in samples]

points_proj['MRT °C'] = points_proj['MRT °C'].astype(str).str.replace(",", ".").astype(float) # ensure numeric without renaming columns
points_proj['MRT_predicted'] = points_proj['MRT_predicted'].astype(float) # if predicted came as float already, ok

# METRICS
obs = points_proj['MRT °C']
pred = points_proj['MRT_predicted']

points_proj['residual'] = pred - obs

# export gpkg
out_gpkg = os.path.join(base_dir, 'MRT-prediction/MRT_points.gpkg')
points_proj.to_file(out_gpkg, driver="GPKG")

In [ ]:
points_proj

In [ ]:

mae = np.mean(np.abs(obs - pred))
rmse = np.sqrt(np.mean((obs - pred)**2))
bias = np.mean(pred - obs)

print("MAE:", mae)
print("RMSE:", rmse)
print("Bias:", bias)

In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(obs, pred)

# 1:1 line from data range
m = min(obs.min(), pred.min())
M = max(obs.max(), pred.max())

plt.plot([m, M], [m, M])

plt.xlabel("Observed MRT °C")
plt.ylabel("Predicted MRT °C")
plt.title("points_proj comparison")
plt.axis("equal")
plt.show()


In [ ]:
residuals = obs - pred  # Compute residuals

plt.figure(figsize=(7,7))
plt.scatter(pred, residuals)
plt.axhline(0)  # Zero reference line

plt.xlabel("Predicted MRT °C")
plt.ylabel("Residuals (Obs - Pred)")
plt.title("Residuals vs Predicted")
plt.show()
